# 00 — Setup & Configuration

This notebook configures the connection to your **Microsoft Foundry** project and verifies
connectivity. Run this once before running the other notebooks (`01` → `02` → `03` → `04`).

Configuration is saved to `../workshop_config.json` (repo root) so all notebooks
can load it automatically.

### Prerequisites
- Python 3.10+
- Azure CLI installed and logged in (`az login`)
- A Microsoft Foundry project with a deployed model (e.g., `gpt-4.1-mini`)

In [ ]:
# Install dependencies
%pip install -q -r requirements.txt
# use uv if available for faster installs
# %uv add -r requirements.txt

In [1]:
# Foundry Connection
#
# Edit the 4 variables below to connect to your Foundry resource.
# If you've already run this notebook (or multi_agent/00-setup), the saved config
# will be loaded automatically.

import json
from pathlib import Path

# ──────────────────────────────────────────────────────────────
# ✏️  Edit these to connect to your Foundry resource
# ──────────────────────────────────────────────────────────────
RESOURCE_GROUP = ""           # e.g. "rg-my-foundry"
ACCOUNT_NAME = ""             # e.g. "my-foundry-account"
PROJECT_NAME = ""             # e.g. "my-project"
MODEL_DEPLOYMENT_NAME = ""    # e.g. "gpt-4.1-mini"
# ──────────────────────────────────────────────────────────────

# Load saved config from repo root if variables are empty
config_path = Path("../workshop_config.json")
if config_path.exists() and not ACCOUNT_NAME:
    with open(config_path) as f:
        config = json.load(f)
    RESOURCE_GROUP = config.get("RESOURCE_GROUP", RESOURCE_GROUP)
    ACCOUNT_NAME = config.get("ACCOUNT_NAME", ACCOUNT_NAME)
    PROJECT_NAME = config.get("PROJECT_NAME", PROJECT_NAME)
    MODEL_DEPLOYMENT_NAME = config.get("MODEL_DEPLOYMENT_NAME", MODEL_DEPLOYMENT_NAME)
    print(f"✅ Loaded saved config from {config_path}")

if not ACCOUNT_NAME:
    raise ValueError(
        "No Foundry connection configured. "
        "Edit the 4 variables above (RESOURCE_GROUP, ACCOUNT_NAME, PROJECT_NAME, MODEL_DEPLOYMENT_NAME)."
    )

PROJECT_ENDPOINT = f"https://{ACCOUNT_NAME}.services.ai.azure.com/api/projects/{PROJECT_NAME}"

print(f"Resource Group:      {RESOURCE_GROUP}")
print(f"Account Name:        {ACCOUNT_NAME}")
print(f"Project Name:        {PROJECT_NAME}")
print(f"Model Deployment:    {MODEL_DEPLOYMENT_NAME}")
print(f"Project Endpoint:    {PROJECT_ENDPOINT}")

✅ Loaded saved config from ..\workshop_config.json
Resource Group:      rg-foundry-iac-ops
Account Name:        fndryiac2ttkx3
Project Name:        iac-ops-demo
Model Deployment:    gpt-4.1-mini
Project Endpoint:    https://fndryiac2ttkx3.services.ai.azure.com/api/projects/iac-ops-demo


In [2]:
# Verify Foundry connectivity

from agent_framework import Agent
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential

credential = AzureCliCredential()

af_client = FoundryChatClient(
    project_endpoint=PROJECT_ENDPOINT,
    model=MODEL_DEPLOYMENT_NAME,
    credential=credential,
)

_test_agent = Agent(client=af_client, instructions="Reply with only the word 'OK'.", name="ConnTest")
_test_resp = await _test_agent.run("ping")
print(f"✅ Foundry connection verified: {_test_resp.text[:50]}")
del _test_agent, _test_resp

<frozen abc>:106: ExperimentalWarning: [HARNESS] MemoryStore is experimental and may change or be removed in future versions without notice.
<frozen abc>:106: ExperimentalWarning: [SKILLS] SkillResource is experimental and may change or be removed in future versions without notice.


✅ Foundry connection verified: OK


### Deploy Required Models

The notebooks in this workshop use two models:
- **gpt-4.1-mini** — primary model for most agents
- **gpt-5.4-mini** — used for evaluation and advanced scenarios

The cell below checks whether these deployments exist in your Foundry project and creates them if missing.

In [ ]:
# Deploy gpt-4.1-mini and gpt-5.4-mini if not already present

from azure.ai.projects import AIProjectClient

project_client = AIProjectClient(endpoint=PROJECT_ENDPOINT, credential=credential)

REQUIRED_MODELS = {
    "gpt-4.1-mini": "gpt-4.1-mini",    # deployment_name: model_name
    "gpt-5.4-mini": "gpt-5.4-mini",
}

# List existing deployments
existing_deployments = {
    d.name for d in project_client.deployments.list()
}
print(f"Existing deployments: {sorted(existing_deployments)}\n")

for deployment_name, model_name in REQUIRED_MODELS.items():
    if deployment_name in existing_deployments:
        print(f"✅ {deployment_name} — already deployed")
    else:
        print(f"⏳ Deploying {deployment_name} (model: {model_name})...")
        project_client.deployments.begin_create_or_update(
            deployment_name=deployment_name,
            deployment={
                "model": {"name": model_name, "version": "latest"},
                "sku": {"name": "GlobalStandard", "capacity": 50},
            },
        ).result()
        print(f"✅ {deployment_name} — deployed successfully")

print(f"\n✅ Both models ready: gpt-4.1-mini, gpt-5.4-mini")

In [3]:
# Save config for other notebooks

config = {
    "RESOURCE_GROUP": RESOURCE_GROUP,
    "ACCOUNT_NAME": ACCOUNT_NAME,
    "PROJECT_NAME": PROJECT_NAME,
    "MODEL_DEPLOYMENT_NAME": MODEL_DEPLOYMENT_NAME,
    "PROJECT_ENDPOINT": PROJECT_ENDPOINT,
}

config_path = Path("../workshop_config.json")
with open(config_path, "w") as f:
    json.dump(config, f, indent=2)

print(f"✅ Config saved to {config_path}")
print("   Run 01-first-agent.ipynb next.")

✅ Config saved to ..\workshop_config.json
   Run 01-first-agent.ipynb next.
